In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql.functions import col, count, when, regexp_extract, trim, lit
from os import truncate


## 1. Data Loading
In this section, the dataset used in the project are loaded.
The data consists of two main sources:
- a dataset containing book metadata
- a dataset containing user reviews and ratings.

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("Books EDA")
         .config("spark.driver.memory", "8g")
         .getOrCreate())

DATA_PATH = "/content/drive/MyDrive/Data_mining/data"

spark_books_df = spark.read.csv(
    f"{DATA_PATH}/books_data.csv",
    header=True,
    inferSchema=True
)

spark_ratings_df = spark.read.csv(
    f"{DATA_PATH}/Books_rating.csv",
    header=True,
    inferSchema=True
)


## 2. Data Understanding

The goal of this step is to gain an understaing of the datasets before performing cleaning and transformations.

In [ ]:
print("BOOKS:")
print("Rows:", spark_books_df.count())
print("Columns:", len(spark_books_df.columns))

print("\nRATINGS:")
print("Rows:", spark_ratings_df.count())
print("Columns:", len(spark_ratings_df.columns))


BOOKS:
Rows: 212404
Columns: 10

RATINGS:
Rows: 3000000
Columns: 10


In [ ]:
spark_books_df.show(5, truncate=False)
spark_ratings_df.show(5, truncate=False)


+-------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
spark_books_df.printSchema()
spark_ratings_df.printSchema()


root
 |-- Title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- image: string (nullable = true)
 |-- previewLink: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- publishedDate: string (nullable = true)
 |-- infoLink: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- ratingsCount: string (nullable = true)

root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/score: string (nullable = true)
 |-- review/time: string (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/text: string (nullable = true)



In [ ]:
spark_books_df.dtypes

[('Title', 'string'),
 ('description', 'string'),
 ('authors', 'string'),
 ('image', 'string'),
 ('previewLink', 'string'),
 ('publisher', 'string'),
 ('publishedDate', 'string'),
 ('infoLink', 'string'),
 ('categories', 'string'),
 ('ratingsCount', 'string')]

In [ ]:
spark_ratings_df.dtypes

[('Id', 'string'),
 ('Title', 'string'),
 ('Price', 'string'),
 ('User_id', 'string'),
 ('profileName', 'string'),
 ('review/helpfulness', 'string'),
 ('review/score', 'string'),
 ('review/time', 'string'),
 ('review/summary', 'string'),
 ('review/text', 'string')]

In [ ]:
total_books = spark_books_df.count()
unique_titles = spark_books_df.select("Title").distinct().count()

print("Total rows:", total_books)
print("Unique titles:", unique_titles)
print("Duplicates:", total_books - unique_titles)


Total rows: 212404
Unique titles: 212400
Duplicates: 4


In [ ]:
total_ratings = spark_ratings_df.count()
unique_ratings = spark_ratings_df.dropDuplicates().count()

print("Total rows:", total_ratings)
print("Unique rows:", unique_ratings)
print("Duplicate rows:", total_ratings - unique_ratings)


Total rows: 3000000
Unique rows: 2991135
Duplicate rows: 8865


## 3. Missing Values

This section examines the extend of missing values across all variables.

In [ ]:
def missing_values(df):
    return df.select([
        count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
        for c in df.columns
    ])

missing_values(spark_books_df).show(truncate=False)
missing_values(spark_ratings_df).show(truncate=False)


+-----+-----------+-------+-----+-----------+---------+-------------+--------+----------+------------+
|Title|description|authors|image|previewLink|publisher|publishedDate|infoLink|categories|ratingsCount|
+-----+-----------+-------+-----+-----------+---------+-------------+--------+----------+------------+
|1    |68357      |31251  |51191|24055      |73130    |25844        |24301   |40524     |148552      |
+-----+-----------+-------+-----+-----------+---------+-------------+--------+----------+------------+

+---+-----+-------+-------+-----------+------------------+------------+-----------+--------------+-----------+
|Id |Title|Price  |User_id|profileName|review/helpfulness|review/score|review/time|review/summary|review/text|
+---+-----+-------+-------+-----------+------------------+------------+-----------+--------------+-----------+
|0  |208  |2517579|562250 |562200     |367               |130         |27         |65            |43         |
+---+-----+-------+-------+-----------+-

## 4. Data Cleaning

- Removal of records missing essential identifiers or target variables.
- Exclusion of malformed or non-numeric rating values.


In [ ]:
# dropping: image, previewLink, infoLink, ratingsCount, description from spark_books_df
books_df_sel = spark_books_df.drop(
    "image",
    "previewLink",
    "infoLink",
    "ratingsCount",
    "description"
)



In [ ]:
# dropping profileName, review/helpfulness, review/time from spark_ratings_df
ratings_df_sel = spark_ratings_df.drop(
    "profileName",
    "review/helpfulness",
    "review/time"
)


In [ ]:
print("=== books_df_sel schema ===")
books_df_sel.printSchema()
books_df_sel.show(5, truncate=False)

print("=== ratings_df_sel schema ===")
ratings_df_sel.printSchema()
ratings_df_sel.show(5, truncate=False)


=== books_df_sel schema ===
root
 |-- Title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- publishedDate: string (nullable = true)
 |-- categories: string (nullable = true)

+-------------------------------------------------------------------------+-------------------------------------------+-------------------------------------------------------------------------------------------+--------------+--------------------------------------------------------------------------------------------------------------------+
|Title                                                                    |authors                                    |publisher                                                                                  |publishedDate |categories                                                                                                          |
+-------------------------------------------------------------------------

In [ ]:
#cleaning categories and publishedDate from malformed data

books_df_sel = books_df_sel.filter(
    ~col("categories").rlike("http")
)
books_df_sel = books_df_sel.filter(
    col("publishedDate").rlike("^[0-9]{4}")
)

In [ ]:
def missing_values(df):
    return df.select([
        count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
        for c in df.columns
    ])

print("=== Missing: books_df_sel ===")
missing_values(books_df_sel).show(truncate=False)

print("=== Missing: ratings_df_sel ===")
missing_values(ratings_df_sel).show(truncate=False)


=== Missing: books_df_sel ===
+-----+-------+---------+-------------+----------+
|Title|authors|publisher|publishedDate|categories|
+-----+-------+---------+-------------+----------+
|1    |31251  |73130    |25844        |40524     |
+-----+-------+---------+-------------+----------+

=== Missing: ratings_df_sel ===
+---+-----+-------+-------+------------+--------------+-----------+
|Id |Title|Price  |User_id|review/score|review/summary|review/text|
+---+-----+-------+-------+------------+--------------+-----------+
|0  |208  |2517579|562250 |130         |65            |43         |
+---+-----+-------+-------+------------+--------------+-----------+



In [ ]:
books_df_sel = books_df_sel.filter(col("Title").isNotNull())
ratings_df_sel = ratings_df_sel.filter(col("Title").isNotNull())
ratings_df_sel = ratings_df_sel.filter(col("review/score").isNotNull())
ratings_df_sel = ratings_df_sel.filter(col("review/text").isNotNull())
ratings_df_sel = ratings_df_sel.filter(col("review/summary").isNotNull())

In [ ]:
missing_values(books_df_sel).show()
missing_values(ratings_df_sel).show()


+-----+-------+---------+-------------+----------+
|Title|authors|publisher|publishedDate|categories|
+-----+-------+---------+-------------+----------+
|    0|   5213|    36552|            0|         0|
+-----+-------+---------+-------------+----------+

+---+-----+-------+-------+------------+--------------+-----------+
| Id|Title|  Price|User_id|review/score|review/summary|review/text|
+---+-----+-------+-------+------------+--------------+-----------+
|  0|    0|2517335| 562219|           0|             0|          0|
+---+-----+-------+-------+------------+--------------+-----------+



In [ ]:
print("=== books_df_sel schema ===")
books_df_sel.printSchema()
books_df_sel.show(5, truncate=False)

print("=== ratings_df_sel schema ===")
ratings_df_sel.printSchema()
ratings_df_sel.show(5, truncate=False)


=== books_df_sel schema ===
root
 |-- Title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- publishedDate: string (nullable = true)
 |-- categories: string (nullable = true)

+-------------------------------------------------------+------------------------+--------------------------+-------------+-----------------------------+
|Title                                                  |authors                 |publisher                 |publishedDate|categories                   |
+-------------------------------------------------------+------------------------+--------------------------+-------------+-----------------------------+
|Its Only Art If Its Well Hung!                         |['Julie Strain']        |NULL                      |1996         |['Comics & Graphic Novels']  |
|Wonderful Worship in Smaller Churches                  |['David R. Ray']        |NULL                      |2000         |['Religion']           

In [ ]:
ratings_df_sel.select("review/score")\
    .where(col("review/score").isNotNull())\
    .groupBy("review/score")\
    .count()\
    .orderBy(col("count").desc())\
    .show(100, truncate=False)


+-------------------+-------+
|review/score       |count  |
+-------------------+-------+
|5.0                |1795795|
|4.0                |581728 |
|3.0                |252940 |
|1.0                |201000 |
|2.0                |150449 |
|0/0                |3226   |
|1/1                |1553   |
|2/2                |878    |
|3/3                |532    |
|0/1                |438    |
|1/2                |416    |
|4/4                |354    |
|2/3                |298    |
| the Austen list"""|268    |
|3/4                |256    |
|5/5                |248    |
|..."               |225    |
|fur-mom"""         |193    |
|4/5                |165    |
|2/4                |164    |
|6/6                |158    |
|1/3                |144    |
|7/7                |139    |
|5/6                |131    |
| for readers"""    |125    |
|0/2                |119    |
|6/7                |103    |
|8/8                |96     |
|3/5                |93     |
|2/5                |88     |
|7/8      

In [ ]:
ratings_df_sel = (
    ratings_df_sel
    .withColumn(
        "score_str",
        when(trim(col("review/score")) == "", lit(None))
        .otherwise(col("review/score"))
    )
    .withColumn(
        "review_score_num",
        regexp_extract(col("score_str"), r"^([1-5]\.0)$", 1)
    )
    .withColumn(
        "review_score_num",
        when(col("review_score_num") == "", lit(None))
        .otherwise(col("review_score_num").cast("double"))
    )
)


In [ ]:
ratings_df_sel.select(
    count("*").alias("total_rows"),
    count(when(col("review_score_num").isNotNull(), True)).alias("valid_scores"),
    count(when(col("review_score_num").isNull(), True)).alias("discarded_rows")
).show()


+----------+------------+--------------+
|total_rows|valid_scores|discarded_rows|
+----------+------------+--------------+
|   2999597|     2981669|         17928|
+----------+------------+--------------+



In [ ]:
ratings_df_sel = ratings_df_sel.filter(
    col("review_score_num").isNotNull()
)


In [ ]:
ratings_df_sel = ratings_df_sel.drop("score_str")


In [ ]:
ratings_df_sel.select("review_score_num").describe().show()

ratings_df_sel.groupBy("review_score_num").count() \
    .orderBy("review_score_num") \
    .show()


+-------+------------------+
|summary|  review_score_num|
+-------+------------------+
|  count|           2981669|
|   mean| 4.214252152066511|
| stddev|1.2040107066369965|
|    min|               1.0|
|    max|               5.0|
+-------+------------------+

+----------------+-------+
|review_score_num|  count|
+----------------+-------+
|             1.0| 200992|
|             2.0| 150438|
|             3.0| 252930|
|             4.0| 581698|
|             5.0|1795611|
+----------------+-------+



In [ ]:

total = ratings_df_sel.count()

ratings_df_sel.groupBy("review_score_num").count() \
    .withColumn("percentage", col("count") / total * 100) \
    .orderBy("review_score_num") \
    .show()


+----------------+-------+------------------+
|review_score_num|  count|        percentage|
+----------------+-------+------------------+
|             1.0| 200992| 6.740922617500466|
|             2.0| 150438| 5.045429254555083|
|             3.0| 252930| 8.482832936855164|
|             4.0| 581698|19.509140685971516|
|             5.0|1795611|60.221674505117775|
+----------------+-------+------------------+



In [ ]:
ratings_df_sel.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Data_mining/clean/ratings_clean.parquet"
)
books_df_sel.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/Data_mining/clean/books_clean.parquet"
)